# 06 — Incertidumbre de las brechas de equidad

**Tesis:** Estratificación de riesgo clínico consciente de la equidad — sesgo geográfico entre entidades.

Los notebooks 03 y 04 reportan las brechas como estimaciones puntuales sobre un único conjunto de prueba. Con 32 entidades de tamaños muy desiguales, parte de la dispersión observada puede ser varianza de muestreo y no inequidad real. Este notebook resuelve esa limitación con dos procedimientos complementarios:

1. **Remuestreo (*bootstrap*) estratificado por entidad**: intervalos de confianza al 95 % para el AUROC y la sensibilidad de cada entidad, y para las brechas.
2. **Prueba de permutación**: contrasta la hipótesis nula de que la entidad no tiene efecto, reasignando al azar las etiquetas de entidad y recalculando la brecha. Es lo que permite afirmar —o no— que las disparidades son *estadísticamente significativas*, que es la formulación original de la hipótesis H1.

> La brecha máxima-menos-mínima es un estadístico sesgado al alza: incluso sin efecto de entidad, con 32 grupos el máximo y el mínimo se separan por azar. Por eso no basta con un intervalo de confianza sobre la brecha; hace falta la distribución nula, que es lo que da la permutación.

## 1. Reconstrucción del escenario auditado

In [1]:
import json

import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

import tesis_common as tc

# Se reproduce exactamente el escenario del notebook 03: mismo modelo, misma
# partición, mismo umbral. No se reentrena nada.
df, manifiesto = tc.cargar_cohorte()
paquete = tc.cargar_modelo(n_filas_cohorte=len(df))

X = df[tc.PREDICTORES].astype("float32")
y = df[tc.TARGET].astype(int)
ent = df["entidad_nombre"]

_, _, idx_te = tc.particionar(y)
p = tc.puntuar(paquete, X.iloc[idx_te])
ya = y.iloc[idx_te].to_numpy()
ea = ent.iloc[idx_te].to_numpy()
THR, _, _ = tc.umbral_para_sensibilidad(ya, p)

ENTS, _ = tc.grupos_evaluables(ea, ya)
IDX = {g: np.where(ea == g)[0] for g in ENTS}
print(f"Test: {len(ya):,} | entidades: {len(ENTS)} | umbral {THR:.4f}")
print(f"AUROC global {roc_auc_score(ya, p):.4f}")

Test: 631,663 | entidades: 32 | umbral 0.1562
AUROC global 0.9503


## 2. Intervalos de confianza por remuestreo

Se remuestrea **dentro de cada entidad**, con reemplazo y conservando su tamaño. Así los intervalos reflejan la incertidumbre del desempeño de cada estado, que es lo que interesa: las entidades pequeñas deben salir con intervalos anchos.

In [2]:
B = 1000
rng = np.random.default_rng(tc.SEMILLA)

def auroc_tpr(yy, pp):
    """AUROC y sensibilidad al umbral fijo; NaN si la réplica quedó degenerada."""
    if len(np.unique(yy)) < 2:
        return np.nan, np.nan
    tpr, _ = tc.tasas(yy, pp, THR)
    return roc_auc_score(yy, pp), tpr

boot_auroc = {g: np.empty(B) for g in ENTS}
boot_tpr = {g: np.empty(B) for g in ENTS}
for b in range(B):
    for g in ENTS:
        ix = IDX[g]
        s = rng.integers(0, len(ix), len(ix))
        a_, t_ = auroc_tpr(ya[ix[s]], p[ix[s]])
        boot_auroc[g][b] = a_
        boot_tpr[g][b] = t_
    if (b + 1) % 250 == 0:
        print(f"  {b+1}/{B} réplicas", flush=True)

gap_auroc_b = np.array([max(boot_auroc[g][b] for g in ENTS) - min(boot_auroc[g][b] for g in ENTS)
                        for b in range(B)])
gap_tpr_b = np.array([max(boot_tpr[g][b] for g in ENTS) - min(boot_tpr[g][b] for g in ENTS)
                      for b in range(B)])

filas = []
for g in ENTS:
    ix = IDX[g]
    a_obs, t_obs = auroc_tpr(ya[ix], p[ix])
    filas.append({"entidad": g, "n": len(ix),
                  "AUROC": a_obs,
                  "AUROC_lo": float(np.nanpercentile(boot_auroc[g], 2.5)),
                  "AUROC_hi": float(np.nanpercentile(boot_auroc[g], 97.5)),
                  "TPR": t_obs,
                  "TPR_lo": float(np.nanpercentile(boot_tpr[g], 2.5)),
                  "TPR_hi": float(np.nanpercentile(boot_tpr[g], 97.5))})
IC = pd.DataFrame(filas).sort_values("AUROC")
IC["amplitud_IC"] = IC["AUROC_hi"] - IC["AUROC_lo"]
print("\n== 5 entidades con AUROC mas bajo ==")
print(IC.head(5).round(4).to_string(index=False))
print("\n== 5 con el intervalo mas ancho (menor precision) ==")
print(IC.nlargest(5, "amplitud_IC")[["entidad","n","AUROC","AUROC_lo","AUROC_hi","amplitud_IC"]]
      .round(4).to_string(index=False))

  250/1000 réplicas


  500/1000 réplicas


  750/1000 réplicas


  1000/1000 réplicas



== 5 entidades con AUROC mas bajo ==
            entidad     n  AUROC  AUROC_lo  AUROC_hi    TPR  TPR_lo  TPR_hi  amplitud_IC
          CHIHUAHUA 11072 0.9207    0.9125    0.9290 0.6882  0.6542  0.7199       0.0165
            CHIAPAS  3831 0.9265    0.9116    0.9407 0.6624  0.6042  0.7205       0.0290
BAJA CALIFORNIA SUR 11712 0.9272    0.9135    0.9398 0.6833  0.6349  0.7308       0.0263
     AGUASCALIENTES  5498 0.9304    0.9182    0.9418 0.6568  0.6033  0.7047       0.0236
            DURANGO  6297 0.9310    0.9190    0.9427 0.6313  0.5796  0.6800       0.0237

== 5 con el intervalo mas ancho (menor precision) ==
            entidad     n  AUROC  AUROC_lo  AUROC_hi  amplitud_IC
            CHIAPAS  3831 0.9265    0.9116    0.9407       0.0290
BAJA CALIFORNIA SUR 11712 0.9272    0.9135    0.9398       0.0263
           CAMPECHE  4191 0.9544    0.9415    0.9653       0.0238
            DURANGO  6297 0.9310    0.9190    0.9427       0.0237
     AGUASCALIENTES  5498 0.9304    0.9182  

## 3. ¿Qué entidades difieren de forma fiable?

Un criterio conservador y fácil de comunicar: contar cuántos pares de entidades tienen intervalos de confianza que **no se solapan**. Sobre 496 pares posibles, esa proporción indica cuánta de la dispersión observada es distinguible del ruido.

In [3]:
from itertools import combinations

pares = list(combinations(IC["entidad"], 2))
R = IC.set_index("entidad")
sin_solape = [(a, b) for a, b in pares
              if R.loc[a, "AUROC_hi"] < R.loc[b, "AUROC_lo"]
              or R.loc[b, "AUROC_hi"] < R.loc[a, "AUROC_lo"]]
print(f"Pares de entidades: {len(pares)}")
print(f"Con IC de AUROC disjuntos: {len(sin_solape)} ({len(sin_solape)/len(pares):.1%})")

peor, mejor = IC.iloc[0], IC.iloc[-1]
disjuntos_extremos = peor["AUROC_hi"] < mejor["AUROC_lo"]
print(f"\nExtremos: {peor['entidad']} [{peor['AUROC_lo']:.4f}, {peor['AUROC_hi']:.4f}] "
      f"vs {mejor['entidad']} [{mejor['AUROC_lo']:.4f}, {mejor['AUROC_hi']:.4f}]")
print(f"  intervalos disjuntos: {disjuntos_extremos}")

Pares de entidades: 496
Con IC de AUROC disjuntos: 261 (52.6%)

Extremos: CHIHUAHUA [0.9125, 0.9290] vs TABASCO [0.9717, 0.9820]
  intervalos disjuntos: True


## 4. Prueba de permutación

Bajo la hipótesis nula de que la entidad no influye en el desempeño, reasignar al azar las etiquetas de entidad —conservando los tamaños de grupo— debe producir brechas comparables a la observada. La fracción de permutaciones que igualan o superan la brecha real es el valor $p$.

In [4]:
P = 1000
tam = np.array([len(IDX[g]) for g in ENTS])
perm_auroc, perm_tpr = np.empty(P), np.empty(P)
for k in range(P):
    orden = rng.permutation(len(ya))
    cortes = np.cumsum(tam)[:-1]
    a_s, t_s = [], []
    for bloque in np.split(orden, cortes):
        a_, t_ = auroc_tpr(ya[bloque], p[bloque])
        a_s.append(a_); t_s.append(t_)
    perm_auroc[k] = np.nanmax(a_s) - np.nanmin(a_s)
    perm_tpr[k] = np.nanmax(t_s) - np.nanmin(t_s)
    if (k + 1) % 250 == 0:
        print(f"  {k+1}/{P} permutaciones", flush=True)

gap_auroc_obs = IC["AUROC"].max() - IC["AUROC"].min()
gap_tpr_obs = IC["TPR"].max() - IC["TPR"].min()
# Regla de Davison-Hinkley: (#{nulo >= observado} + 1) / (P + 1), nunca p = 0.
p_auroc = (int((perm_auroc >= gap_auroc_obs).sum()) + 1) / (P + 1)
p_tpr = (int((perm_tpr >= gap_tpr_obs).sum()) + 1) / (P + 1)

print(f"\nBrecha de AUROC observada {gap_auroc_obs:.4f} | nula: mediana "
      f"{np.median(perm_auroc):.4f}, p95 {np.percentile(perm_auroc, 95):.4f} -> p = {p_auroc:.4f}")
print(f"Brecha de TPR   observada {gap_tpr_obs:.4f} | nula: mediana "
      f"{np.median(perm_tpr):.4f}, p95 {np.percentile(perm_tpr, 95):.4f} -> p = {p_tpr:.4f}")
print("\n-> la brecha nula NO es cero: con 32 grupos, maximo y minimo se separan por azar.")
print("   Lo que importa es cuanto excede la observada a esa referencia.")

  250/1000 permutaciones


  500/1000 permutaciones


  750/1000 permutaciones


  1000/1000 permutaciones



Brecha de AUROC observada 0.0563 | nula: mediana 0.0175, p95 0.0252 -> p = 0.0010
Brecha de TPR   observada 0.2935 | nula: mediana 0.0737, p95 0.1062 -> p = 0.0010

-> la brecha nula NO es cero: con 32 grupos, maximo y minimo se separan por azar.
   Lo que importa es cuanto excede la observada a esa referencia.


## 5. Figura y exportación

In [5]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(15, 8))
o = IC.sort_values("AUROC")
ypos = np.arange(len(o))
ax[0].errorbar(o["AUROC"], ypos,
               xerr=[o["AUROC"] - o["AUROC_lo"], o["AUROC_hi"] - o["AUROC"]],
               fmt="o", color="#1F3864", ecolor="#8EA9DB", capsize=2, markersize=4)
ax[0].set_yticks(ypos); ax[0].set_yticklabels(o["entidad"], fontsize=7)
ax[0].set_xlabel("AUROC (IC 95 % por remuestreo)")
ax[0].set_title("Desempeño por entidad con incertidumbre")

ax[1].hist(perm_auroc, bins=40, color="#BFBFBF", label="brecha bajo la nula")
ax[1].axvline(gap_auroc_obs, color="#C00000", lw=2, label="brecha observada")
ax[1].set_xlabel("Brecha de AUROC entre entidades")
ax[1].set_ylabel("Permutaciones")
ax[1].set_title(f"Prueba de permutación (p = {p_auroc:.4f})")
ax[1].legend()
plt.tight_layout(); plt.savefig("incertidumbre_brechas.png", dpi=110)
print("Figura guardada: incertidumbre_brechas.png")

salida = {
    "B_bootstrap": B, "P_permutaciones": P,
    "gap_auroc_obs": round(float(gap_auroc_obs), 4),
    "gap_tpr_obs": round(float(gap_tpr_obs), 4),
    "gap_auroc_ic": [round(float(np.percentile(gap_auroc_b, 2.5)), 4),
                     round(float(np.percentile(gap_auroc_b, 97.5)), 4)],
    "gap_tpr_ic": [round(float(np.percentile(gap_tpr_b, 2.5)), 4),
                   round(float(np.percentile(gap_tpr_b, 97.5)), 4)],
    "nula_auroc_mediana": round(float(np.median(perm_auroc)), 4),
    "nula_auroc_p95": round(float(np.percentile(perm_auroc, 95)), 4),
    "nula_tpr_mediana": round(float(np.median(perm_tpr)), 4),
    "nula_tpr_p95": round(float(np.percentile(perm_tpr, 95)), 4),
    "p_auroc": round(float(p_auroc), 4), "p_tpr": round(float(p_tpr), 4),
    "pares_total": len(pares), "pares_disjuntos": len(sin_solape),
    "frac_pares_disjuntos": round(len(sin_solape) / len(pares), 4),
    "extremos_disjuntos": bool(disjuntos_extremos),
    "ic_mas_ancho": IC.nlargest(1, "amplitud_IC")["entidad"].iloc[0],
    "ic_mas_ancho_amplitud": round(float(IC["amplitud_IC"].max()), 4),
    "ic_mas_ancho_n": int(IC.nlargest(1, "amplitud_IC")["n"].iloc[0]),
    "por_entidad": IC.round(4).to_dict(orient="records"),
}
with open("incertidumbre_brechas.json", "w", encoding="utf-8") as fh:
    json.dump(salida, fh, indent=2, ensure_ascii=False)
print("Exportado: incertidumbre_brechas.json")

Figura guardada: incertidumbre_brechas.png
Exportado: incertidumbre_brechas.json


## 6. Lectura

El intervalo de confianza de cada entidad indica con qué precisión se estima su desempeño; las entidades pequeñas tendrán intervalos anchos y su posición en el ordenamiento no debe interpretarse.

La prueba de permutación responde la pregunta que la formulación de H1 planteaba y que hasta ahora quedaba sin contestar: si la dispersión observada entre entidades excede la que produciría el azar con grupos de esos tamaños.